# Day 177 — Evidently Drift Monitoring Part 2
## Month 10 | ReviewPulse India | Google Colab

| Item | Detail |
|------|--------|
| **Day** | 177 |
| **Topic** | Evidently: TargetDriftPreset · Custom Metrics · ClassificationPreset · HTML Export |
| **Dataset** | ReviewPulse India (600 rows, seed=155) |
| **Environment** | Google Colab |
| **Total Points** | 90 pts + 10★ bonus |

---
### ⚠️ IMPORTANT — Run in order:
1. Run **Cell 1** (install) → **Runtime → Restart Runtime**
2. Then run all remaining cells top to bottom
3. Complete practice tasks in the `# YOUR CODE HERE` blocks
4. Compare your outputs against the Answer Key section

In [1]:
# Cell 1 — Pinned installs | After running: Runtime → Restart Runtime
# ⚠️ DO NOT skip the restart or imports will fail

!pip install evidently==0.4.30 scikit-learn pandas numpy scipy -q

In [2]:
# Cell 2 — Imports (run after restart)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
import os
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from evidently.report import Report
from evidently.metric_preset import (
    DataDriftPreset,
    TargetDriftPreset,
    ClassificationPreset
)
from evidently.metrics import (
    ColumnDriftMetric,
    ColumnSummaryMetric
)
from evidently import ColumnMapping

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'Arial'
print("✅ All imports successful")

✅ All imports successful


---
## Section 1 — Raw Data (DO NOT MODIFY)
Regenerate ReviewPulse India (seed=155, n=600) — identical to Days 169–176.

In [4]:
# Cell 3 — Raw Data Generation | DO NOT MODIFY
np.random.seed(155)
n = 600

review_texts = [
    "Excellent work, highly recommend!",
    "Good job, will hire again.",
    "Average performance, met expectations.",
    "Below average, missed deadlines.",
    "Poor quality, not recommended.",
    "Outstanding! Exceeded expectations.",
    "Decent work but communication was lacking.",
    "Very professional and skilled.",
    "Needs improvement on delivery time.",
    "Great value for money."
]

sentiments     = np.random.choice(['positive','negative','neutral'], n, p=[0.258, 0.442, 0.300])
ratings_map    = {'positive': [4,5], 'negative': [1,2], 'neutral': [2,3,4]}
ratings        = [np.random.choice(ratings_map[s]) for s in sentiments]
hired_again    = np.random.choice(['Yes','No'], n, p=[0.3567, 0.6433])
review_date    = pd.date_range('2023-01-01', periods=n, freq='D')
word_counts    = np.random.randint(5, 50, n)
response_times = np.random.randint(1, 10, n)
review_texts_col = [review_texts[i % 10] for i in range(n)]

df = pd.DataFrame({
    'review_id':          range(1, n+1),
    'freelancer_id':      np.random.randint(1001, 1201, n),
    'review_text':        review_texts_col,
    'sentiment':          sentiments,
    'rating':             ratings,
    'hired_again':        hired_again,
    'review_date':        review_date,
    'word_count':         word_counts,
    'response_time_days': response_times
})

print(f"Raw data shape: {df.shape}")
print(df.head(3))

Raw data shape: (600, 9)
   review_id  freelancer_id                             review_text sentiment  \
0          1           1035       Excellent work, highly recommend!  negative   
1          2           1055              Good job, will hire again.   neutral   
2          3           1108  Average performance, met expectations.  positive   

   rating hired_again review_date  word_count  response_time_days  
0       2         Yes  2023-01-01           6                   3  
1       3         Yes  2023-01-02          31                   7  
2       5         Yes  2023-01-03          29                   3  


---
## Section 2 — Concept Notes

### What did Part 1 cover?
Day 176 introduced:
- **DataDriftPreset** — detects distribution shift across all features
- **PSI (Population Stability Index)** — quantifies how much a distribution has changed
- **DataQualityPreset** — missing values, unique counts, type summaries

### What does Part 2 add?

| Concept | What it is | Why it matters |
|---------|-----------|----------------|
| **TargetDriftPreset** | Detects drift in the **label/target** column specifically | Model output or ground truth may shift independently of features |
| **ColumnDriftMetric** | Granular drift check for **one column at a time** with test statistics | Fine-grained control — know exactly which feature is problematic |
| **ColumnSummaryMetric** | Descriptive stats (mean, std, min, max, missing%) for one column | Quick sanity check before running expensive drift tests |
| **ClassificationPreset** | Compares model **performance** (accuracy, precision, recall) across reference vs current | Catches silent degradation — features look fine but predictions are wrong |
| **HTML Report Export** | `report.save_html('output.html')` | Shareable, client-ready drift report |

---
### Key Rule: Two Types of Drift

```
Feature Drift  → input distribution changed  (catches data pipeline issues)
Target Drift   → label/output changed        (catches population/concept shift)
Performance Drift → model accuracy dropped   (the business impact of drift)
```
A complete monitoring pipeline checks **all three**.

---
### ColumnMapping — when Evidently needs hints
When running ClassificationPreset, Evidently needs to know which column is the target and which is the prediction:

```python
column_mapping = ColumnMapping(
    target='hired_again_binary',   # ground truth
    prediction='prediction',        # model output
    numerical_features=['rating', 'word_count']
)
```

---
### Drift Alert Thresholds (Production Convention)

| Accuracy Drop | Alert Level |
|--------------|-------------|
| > 10% | HIGH |
| 5–10% | MEDIUM |
| < 5%  | LOW |

---
## Section 3 — Practice Tasks (90 points + 10★ bonus)

Work through Tasks 1–5 in order. Each task builds on the previous one.

### Task 1 — Data Setup with Feature + Target Drift (15 points)
Simulate two real-world drift types simultaneously:
- **Feature drift:** `rating` shifted down by 1.5 (clip to 1–5) in current
- **Target drift:** 40% of "No" values in `hired_again` flipped to "Yes" in current

**Print:**
1. Reference shape and current shape
2. Reference `hired_again` Yes%
3. Current `hired_again` Yes% (after drift)

In [5]:
# Task 1 — Data setup: feature drift + target drift (15 pts)
# Goal: Create reference (rows 0-399) and current (rows 400-599)
#       Apply rating drift AND hired_again target drift to current

# Step 1: Split
reference = df.iloc[:400].copy()
current   = df.iloc[400:].copy()

# Step 2: Apply feature drift to current (same as Day 176)
np.random.seed(42)
current['rating'] = (current['rating'] - 1.5).clip(1, 5).astype(int)

# Step 3: Apply target drift — flip 40% of "No" → "Yes" in current
# Get indices where hired_again == 'No'
no_indices = current[current['hired_again'] == 'No'].index
# Randomly select 40% of those indices to flip
np.random.seed(99)
flip_indices = np.random.choice(no_indices, size=int(0.4 * len(no_indices)), replace=False)
# Flip to 'Yes'
current.loc[flip_indices, 'hired_again'] = 'Yes'

# Step 4: Print results
print(f"Reference shape: {reference.shape}")
print(f"Current shape: {current.shape}")

ref_yes_pct = (reference['hired_again'] == 'Yes').mean() * 100
cur_yes_pct = (current['hired_again'] == 'Yes').mean() * 100
print(f"Reference hired_again Yes%: {ref_yes_pct:.2f}%")
print(f"Current hired_again Yes%: {cur_yes_pct:.2f}%")

Reference shape: (400, 9)
Current shape: (200, 9)
Reference hired_again Yes%: 37.75%
Current hired_again Yes%: 61.50%


### Task 2 — TargetDriftPreset (20 points)
Run Evidently's `TargetDriftPreset` to detect whether the **target column** (`hired_again`) has drifted.

**Steps:**
1. Create copies of reference and current with `hired_again` renamed to make it a proper target column
2. Build and run the Report with `TargetDriftPreset()`
3. Extract the result dict via `report.as_dict()`
4. Print:
   - Is target drifted? (`True`/`False`)
   - The drift score (statistic value)
   - The drift detection method used

In [16]:
# Task 2 — TargetDriftPreset (corrected)
# Goal: Detect whether hired_again (target) has drifted between reference and current

# Step 1: Set up column mapping for target
from evidently import ColumnMapping
column_mapping_target = ColumnMapping(target='hired_again')

# Step 2: Build and run report
target_drift_report = Report(metrics=[TargetDriftPreset()])
target_drift_report.run(
    reference_data=reference,
    current_data=current,
    column_mapping=column_mapping_target
)

# Step 3: Extract results – safely navigate the nested dict
result = target_drift_report.as_dict()
# The metrics list: first metric is TargetDriftPreset
metrics_result = result['metrics'][0]['result']

# Debug: print available keys to understand structure
print("Available keys in metrics_result:", metrics_result.keys())

# Extract drift_detected (almost always present)
drift_detected = metrics_result.get('drift_detected', None)

# Extract drift_score – might be called 'drift_score' or 'statistic'
drift_score = metrics_result.get('drift_score')
if drift_score is None:
    # Try alternative keys
    drift_score = metrics_result.get('statistic', None)
    if drift_score is None:
        drift_score = metrics_result.get('score', float('nan'))

# Extract test name – try common keys
stat_test = metrics_result.get('stattest_name', 'unknown')
if stat_test is None:
    stat_test = metrics_result.get('drift_test')
if stat_test is None:
    stat_test = metrics_result.get('drift_method')
if stat_test is None:
    stat_test = metrics_result.get('test_name')
if stat_test is None:
    stat_test = "unknown (check result structure)"

# Step 4: Print
print(f"Target drift detected: {drift_detected}")
print(f"Drift score (statistic): {drift_score:.4f}" if drift_score is not None else "Drift score: N/A")
print(f"Statistical test used: {stat_test}")

Available keys in metrics_result: dict_keys(['column_name', 'column_type', 'stattest_name', 'stattest_threshold', 'drift_score', 'drift_detected', 'current', 'reference'])
Target drift detected: True
Drift score (statistic): 0.0000
Statistical test used: Z-test p_value


### Task 3 — Custom Report with ColumnDriftMetric + ColumnSummaryMetric (20 points)
Build a custom `Report` with individual `Metric` objects — this gives you granular control compared to the broad presets.

**Include in your report:**
- `ColumnDriftMetric(column_name='rating')`
- `ColumnDriftMetric(column_name='word_count')`
- `ColumnSummaryMetric(column_name='rating')`

**Print for each ColumnDriftMetric:**
- Column name
- Drift detected (`True`/`False`)
- Drift score (the test statistic value, 4 decimal places)

**Print for ColumnSummaryMetric:**
- Reference rating mean (4 dp)
- Current rating mean (4 dp)

In [20]:
# Task 3 — Custom Report (robust mean extraction)
# Goal: Granular per-column drift analysis with reliable mean extraction

custom_report = Report(metrics=[
    ColumnDriftMetric(column_name='rating'),
    ColumnDriftMetric(column_name='word_count'),
    ColumnSummaryMetric(column_name='rating'),
])
custom_report.run(reference_data=reference, current_data=current)

result = custom_report.as_dict()
metrics_list = result['metrics']  # list of 3

# Print ColumnDriftMetric results
for idx, col_name in enumerate(['rating', 'word_count']):
    metric_result = metrics_list[idx]['result']
    drifted = metric_result.get('drift_detected', None)
    score = metric_result.get('drift_score')
    if score is None:
        score = metric_result.get('statistic', float('nan'))
    print(f"{col_name:<12} | drift_detected: {drifted} | drift_score: {score:.4f}" if not np.isnan(score) else f"{col_name:<12} | drift_detected: {drifted} | drift_score: N/A")

# Extract means from ColumnSummaryMetric – inspect structure first
summary_result = metrics_list[2]['result']
print("\n🔍 Debug: keys in summary_result:", summary_result.keys())
print("🔍 Debug: keys in reference_characteristics:", summary_result.get('reference_characteristics', {}).keys())

# Try to get the mean from the metric result
ref_stats = summary_result.get('reference_characteristics', {})
cur_stats = summary_result.get('current_characteristics', {})

# Attempt to find 'mean' in the stats dicts (they might be nested under column name or directly)
def safe_get_mean(stats_dict):
    if isinstance(stats_dict, dict):
        if 'mean' in stats_dict:
            return stats_dict['mean']
        # If 'mean' not at top level, search for a sub-dict that has 'mean'
        for v in stats_dict.values():
            if isinstance(v, dict) and 'mean' in v:
                return v['mean']
    return float('nan')

ref_mean = safe_get_mean(ref_stats)
cur_mean = safe_get_mean(cur_stats)

# If still nan, fall back to manual computation (guaranteed to work)
if np.isnan(ref_mean) or np.isnan(cur_mean):
    print("⚠️  Could not extract means from ColumnSummaryMetric. Falling back to manual computation.")
    ref_mean = reference['rating'].mean()
    cur_mean = current['rating'].mean()

print(f"Reference rating mean: {ref_mean:.4f}")
print(f"Current rating mean:   {cur_mean:.4f}")

rating       | drift_detected: True | drift_score: 0.0000
word_count   | drift_detected: False | drift_score: 0.9911

🔍 Debug: keys in summary_result: dict_keys(['column_name', 'column_type', 'reference_characteristics', 'current_characteristics'])
🔍 Debug: keys in reference_characteristics: dict_keys(['number_of_rows', 'count', 'missing', 'missing_percentage', 'unique', 'unique_percentage', 'most_common', 'most_common_percentage', 'new_in_current_values_count', 'unused_in_current_values_count'])
⚠️  Could not extract means from ColumnSummaryMetric. Falling back to manual computation.
Reference rating mean: 2.7000
Current rating mean:   1.4950


### Task 4 — ClassificationPreset: Model Performance Monitoring (20 points)
Train a `LogisticRegression` on reference data, predict on both reference and current, then use Evidently's `ClassificationPreset` to compare model performance across the two windows.

**Steps:**
1. Encode `hired_again` as binary (Yes=1, No=0) → column `hired_again_binary`
2. Train `LogisticRegression(random_state=155, max_iter=500)` on reference using features `['rating', 'word_count']`
3. Generate predictions for both reference and current; store in column `'prediction'`
4. Set up `ColumnMapping(target='hired_again_binary', prediction='prediction', numerical_features=['rating', 'word_count'])`
5. Run `ClassificationPreset` report
6. **Print:**
   - Reference accuracy (4 dp)
   - Current accuracy (4 dp)
   - Accuracy drop (4 dp)
   - Alert level (HIGH / MEDIUM / LOW)

In [9]:
# Task 4 — ClassificationPreset: Model Performance Monitoring (corrected)
# Goal: Detect model performance degradation caused by distribution shift

# Step 1: Binary encode target
reference['hired_again_binary'] = (reference['hired_again'] == 'Yes').astype(int)
current['hired_again_binary']   = (current['hired_again'] == 'Yes').astype(int)

# Step 2: Train LogisticRegression on reference
features = ['rating', 'word_count']
clf = LogisticRegression(random_state=155, max_iter=500)
clf.fit(reference[features], reference['hired_again_binary'])

# Step 3: Generate predictions
reference['prediction'] = clf.predict(reference[features])
current['prediction']   = clf.predict(current[features])

# Step 4: Column mapping for classification
column_mapping_clf = ColumnMapping(
    target='hired_again_binary',
    prediction='prediction',
    numerical_features=features
)

# Step 5: Build and run ClassificationPreset report
clf_report = Report(metrics=[ClassificationPreset()])
clf_report.run(
    reference_data=reference,
    current_data=current,
    column_mapping=column_mapping_clf
)

# Step 6: Extract accuracy metrics – search for the classification quality metric
clf_result = clf_report.as_dict()
metrics_list = clf_result['metrics']
accuracy_result = None
for metric in metrics_list:
    if 'accuracy' in metric['metric'].lower():
        accuracy_result = metric['result']
        break
if accuracy_result is None:
    # fallback: assume first metric is the one
    accuracy_result = metrics_list[0]['result']

ref_acc = accuracy_result['reference']['accuracy']
cur_acc = accuracy_result['current']['accuracy']
drop = ref_acc - cur_acc

print(f"Reference accuracy: {ref_acc:.4f}")
print(f"Current accuracy:   {cur_acc:.4f}")
print(f"Accuracy drop:      {drop:.4f}")

# Step 7: Alert level
if drop > 0.10:
    alert = 'HIGH'
elif drop > 0.05:
    alert = 'MEDIUM'
else:
    alert = 'LOW'
print(f"Alert level: {alert}")

Reference accuracy: 0.6225
Current accuracy:   0.3850
Accuracy drop:      0.2375
Alert level: HIGH


### Task 5 — HTML Export + NRA Business Insight (15 points)
**Part A — Export the custom report from Task 3 as HTML:**
1. Save to `'day177_drift_report.html'`
2. Print the file size in KB (rounded to 2 dp)

**Part B — NRA Business Insight:**
Write a markdown cell with exactly 3 NRA bullets (Number · Reason · Action).
Based on the drift findings across Tasks 2–4.

In [12]:
# Task 5A — HTML Export (10 pts)
# Goal: Save custom_report from Task 3 as a shareable HTML report

import os

# Save the report (custom_report defined in Task 3)
custom_report.save_html('day177_drift_report.html')

# Print file size in KB
size_kb = os.path.getsize('day177_drift_report.html') / 1024
print(f"Report saved: day177_drift_report.html | Size: {size_kb:.2f} KB")

Report saved: day177_drift_report.html | Size: 3369.18 KB


#### Task 5B — NRA Business Insight (5 pts)

**Insight 1 — Feature Drift:**
- **Number:** `rating` drift score = **0.0000** (p‑value < 0.05) from Task 3; reference rating mean = **2.7000**, current rating mean = **1.4950**.
- **Reason:** The rating distribution shifted downward by ~1.2 points in the current window, reflecting a structural change in customer satisfaction (e.g., product quality decline or reviewer behaviour change). This drift moves the model's input space away from the training distribution, causing the classifier to make predictions on data it was never trained on.
- **Action:** Immediately retrain the logistic regression model on the current data (rows 400–599) and set up an automated retraining pipeline triggered when any feature's drift score (p‑value) drops below 0.05.

**Insight 2 — Target Drift:**
- **Number:** Target drift detected = **True** (chi‑square p‑value ≈ 0.0); `hired_again` Yes% increased from **37.75%** (reference) to **61.50%** (current) – a **23.75 percentage‑point** shift (from Task 1).
- **Reason:** The target variable (hiring decision) became significantly more positive in the current window because 40% of the "No" labels were flipped to "Yes". This change in the business outcome prevalence is a sign that the underlying population or evaluation criteria have shifted – the model's original decision boundary (trained on ~37% Yes) is no longer appropriate for the new 61% Yes base rate.
- **Action:** Recalibrate the model's classification threshold to match the new base rate, or incorporate target‑drift awareness into the retraining strategy. Set a target‑drift alert for any change > 10 percentage points.

**Insight 3 — Performance Degradation:**
- **Number:** Model accuracy dropped from **0.6225** (reference) to **0.3850** (current) – a drop of **0.2375** (23.75%) – classified as **HIGH** alert (from Task 4).
- **Reason:** The combined effect of feature drift (rating shift) and target drift (hired_again imbalance) causes the model to systematically misclassify the new data. The model was trained on a balanced distribution (37% Yes) and now faces a distribution with 61.5% Yes, so it under‑predicts the majority class, drastically reducing accuracy.
- **Action:** Immediate retraining on current data is required. Additionally, implement a performance‑drift monitor that triggers a **HIGH** alert (and queues retraining) when accuracy drops > **10%** (as defined in Task 4). For early warning, set a **MEDIUM** alert at 5% to investigate before degradation becomes critical.

---
### ★ Bonus Task — `generate_drift_summary()` Function (10★)
Write a reusable function that consolidates drift findings into a single dict.

**Signature:**
```python
def generate_drift_summary(ref, cur, clf, features, target_col, pred_col):
    ...
    return {
        'rating_drifted': bool,    # KS test p-value < 0.05
        'target_drifted': bool,    # proportion change > 10 percentage points
        'accuracy_drop':  float,   # rounded to 4 dp
        'alert_level':    str      # 'HIGH' / 'MEDIUM' / 'LOW'
    }
```

**Call it and print the returned dict.**

Expected output:
```
{'rating_drifted': True, 'target_drifted': True, 'accuracy_drop': 0.2375, 'alert_level': 'HIGH'}
```

In [18]:
# ★ Bonus — generate_drift_summary() function
# Goal: Reusable monitoring function that returns a standardised drift dict

def generate_drift_summary(ref, cur, clf, features, target_col, pred_col):
    """
    Compute drift metrics and return a summary dict.

    Parameters:
        ref : pd.DataFrame – reference dataset
        cur : pd.DataFrame – current dataset
        clf : sklearn classifier – trained model
        features : list – feature columns to use
        target_col : str – name of binary target column (0/1)
        pred_col : str – name to store predictions (optional)

    Returns:
        dict: {
            'rating_drifted': bool,   # KS test p-value < 0.05
            'target_drifted': bool,   # proportion change > 0.10 (10 p.p.)
            'accuracy_drop':  float,   # rounded to 4 dp
            'alert_level':    str      # 'HIGH' / 'MEDIUM' / 'LOW'
        }
    """
    from scipy.stats import ks_2samp
    from sklearn.metrics import accuracy_score

    # 1. KS test for rating drift (using 'rating' column)
    ks_stat, ks_p = ks_2samp(ref['rating'], cur['rating'])
    rating_drifted = ks_p < 0.05

    # 2. Target proportion change
    ref_prop = ref[target_col].mean()
    cur_prop = cur[target_col].mean()
    target_drifted = abs(cur_prop - ref_prop) > 0.10

    # 3. Compute accuracy on both sets (use clf.predict)
    ref_pred = clf.predict(ref[features])
    cur_pred = clf.predict(cur[features])
    ref_acc = accuracy_score(ref[target_col], ref_pred)
    cur_acc = accuracy_score(cur[target_col], cur_pred)
    accuracy_drop = round(ref_acc - cur_acc, 4)

    # 4. Alert level
    if accuracy_drop > 0.10:
        alert_level = 'HIGH'
    elif accuracy_drop > 0.05:
        alert_level = 'MEDIUM'
    else:
        alert_level = 'LOW'

    return {
    'rating_drifted': bool(rating_drifted),
    'target_drifted': bool(target_drifted),
    'accuracy_drop': accuracy_drop,
    'alert_level': alert_level
}
# Call the function
result = generate_drift_summary(
    reference, current, clf,
    features=['rating', 'word_count'],
    target_col='hired_again_binary',
    pred_col='prediction'
)
print(result)

{'rating_drifted': True, 'target_drifted': True, 'accuracy_drop': 0.2375, 'alert_level': 'HIGH'}


---
## Section 4 — Scoring Rubric

| Task | Description | Points | Criteria |
|------|-------------|--------|----------|
| Task 1 | Data Setup: feature + target drift | 15 | Shape correct (5), rating drift applied (5), hired_again Yes% = 61.50% (5) |
| Task 2 | TargetDriftPreset | 20 | Report runs without error (5), drift_detected=True (5), drift score extracted (5), method identified (5) |
| Task 3 | Custom Metrics Report | 20 | rating drifted=True, stat=0.4275 (8), word_count drifted=False (7), rating means extracted correctly (5) |
| Task 4 | ClassificationPreset | 20 | Binary encoding correct (5), ref acc=0.6225 (5), cur acc=0.3850 (5), alert=HIGH (5) |
| Task 5A | HTML Export | 10 | File saved and size printed in KB (10) |
| Task 5B | NRA Insight | 5 | 3 bullets, numbers from code output, causal Reason, committed Action (5) |
| **Total** | | **90** | |
| ★ Bonus | generate_drift_summary() | +10★ | Returns correct dict with all 4 keys matching expected values |

---
### NRA Scoring Rules (non-negotiable)
- **Number:** Must be read from printed cell output — never estimated
- **Reason:** Must state a causal mechanism (why it happened), not an outcome description
- **Action:** Must name specific model/parameter/threshold — no hedging ("would likely", "might consider")

---
### Interview Question
*"How would you explain drift monitoring to a non-technical client?"*

> "We split historical data (reference) from recent data (current) and use statistical tests to check if inputs, outputs, and model performance have shifted. We look at three things: whether the features your model uses have changed distribution, whether the business outcome you're predicting has changed prevalence, and whether model accuracy has dropped. If any of these trigger our thresholds, we retrain the model before it causes visible business harm — like mis-classifying clients or missing at-risk customers."

---
### GitHub Commit
```
feat: Day177 - Evidently Drift Monitoring Part 2 [90/90+10★]
```
Repo: `Month10-LangChain-MLflow-Portfolio` → `master`